In [8]:
from pathlib import Path

zarr_path = Path('/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr')

source = "swot-river"
source_path = zarr_path / source
if source_path.exists():
    processed = [d.stem for d in source_path.iterdir() if d.is_dir()]
processed

['ABOM-12049010',
 'ABOM-121206010',
 'ABOM-122989010',
 'ABOM-123864010',
 'ABOM-128547010',
 'ABOM-133163010',
 'ABOM-134767010',
 'ABOM-135379010']

In [6]:
list(source_path.iterdir())

[PosixPath('/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr/swot-river/ABOM-12049010'),
 PosixPath('/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr/swot-river/ABOM-121206010'),
 PosixPath('/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr/swot-river/ABOM-122989010'),
 PosixPath('/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr/swot-river/ABOM-123864010'),
 PosixPath('/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr/swot-river/ABOM-128547010'),
 PosixPath('/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr/swot-river/ABOM-133163010'),
 PosixPath('/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr/swot-river/ABOM-134767010'),
 PosixPath('/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr/swot-river/ABOM-135379010'),
 PosixPath('/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr/swot-river/zarr.json')]

In [1]:
import pandas as pd
from data import BasinDataLake

In [2]:
store = BasinDataLake(root_dir="./my_delta_dataset")

In [3]:
pwd

'/nas/cee-water/cjgleason/ted/swot-ml/notebooks/reservoirs/preprocess'

In [10]:
# 2. Append data for multiple subbasins (this creates small files)
dates = pd.to_datetime(pd.date_range("2021-01-01", "2021-01-10", freq="D"))

source_dict = {
    'ERA5': pd.DataFrame(index=dates, data={'precip': range(10)}),
    'SWOT': pd.DataFrame(index=dates, data={'height': range(10)}),
    'S2': pd.DataFrame(index=dates, data={'width': range(10)}),
    'gauge': pd.DataFrame(index=dates, data={'discharge': range(10)}),
}


for basin in ['larry','curly','moe']:
    print(basin)
    for source, data in source_dict.items():
        print(source)
        
        data_dict = {}
        for subbasin in range(6):
            data_dict[f"{basin}_{subbasin}"] = data
    
        store.write_dynamic(basin, source, data_dict, mode='append')

larry
ERA5
SWOT
S2
gauge
curly
ERA5
SWOT
S2
gauge
moe
ERA5
SWOT
S2
gauge


In [4]:
store.is_processed('larry','larry_0','ERA5')

True

In [4]:
store.get_processing_status('larry','ERA5')

,basin,subbasin,source,processed_at,has_data
0,larry,larry_0,ERA5,2025-09-20 16:04:18.075775+00:00,True
1,larry,larry_1,ERA5,2025-09-20 16:04:18.075775+00:00,True
2,larry,larry_2,ERA5,2025-09-20 16:04:18.075775+00:00,True
3,larry,larry_3,ERA5,2025-09-20 16:04:18.075775+00:00,True
4,larry,larry_4,ERA5,2025-09-20 16:04:18.075775+00:00,True
5,larry,larry_5,ERA5,2025-09-20 16:04:18.075775+00:00,True


In [12]:
df.xs('larry_0', level='subbasin')

source,ERA5,SWOT,S2,gauge
variable,precip,height,width,discharge
date,,,,
2021-01-01 00:00:00+00:00,0,0,0,0
2021-01-02 00:00:00+00:00,1,1,1,1
2021-01-03 00:00:00+00:00,2,2,2,2
2021-01-04 00:00:00+00:00,3,3,3,3
2021-01-05 00:00:00+00:00,4,4,4,4
2021-01-06 00:00:00+00:00,5,5,5,5
2021-01-07 00:00:00+00:00,6,6,6,6
2021-01-08 00:00:00+00:00,7,7,7,7


In [11]:
# 3. Read the data - it all works seamlessly
df = store.read_dynamic(basin='larry')#, concat_sources=False)
df

source                               ERA5   SWOT    S2     gauge
variable                           precip height width discharge
subbasin date                                                   
larry_0  2021-01-01 00:00:00+00:00      0      0     0         0
         2021-01-02 00:00:00+00:00      1      1     1         1
         2021-01-03 00:00:00+00:00      2      2     2         2
         2021-01-04 00:00:00+00:00      3      3     3         3
         2021-01-05 00:00:00+00:00      4      4     4         4
...                                   ...    ...   ...       ...
larry_5  2021-01-06 00:00:00+00:00      5      5     5         5
         2021-01-07 00:00:00+00:00      6      6     6         6
         2021-01-08 00:00:00+00:00      7      7     7         7
         2021-01-09 00:00:00+00:00      8      8     8         8
         2021-01-10 00:00:00+00:00      9      9     9         9

[120 rows x 4 columns]

In [11]:
df['source']

date
2021-01-01 00:00:00+00:00     ERA5
2021-01-01 00:00:00+00:00       S2
2021-01-01 00:00:00+00:00       S2
2021-01-01 00:00:00+00:00       S2
2021-01-01 00:00:00+00:00       S2
                             ...  
2021-01-10 00:00:00+00:00       S2
2021-01-10 00:00:00+00:00       S2
2021-01-10 00:00:00+00:00       S2
2021-01-10 00:00:00+00:00     SWOT
2021-01-10 00:00:00+00:00    gauge
Name: source, Length: 240, dtype: object

In [ ]:
store.compact()

In [ ]:
store.vacuum(retention_hours=0) 

In [ ]:
store.table_uri